# MultiMedAI — Fine-tune Stable Diffusion v1.5 (LoRA) on a FREE GPU

**Why this notebook exists:** your local machine is AMD Radeon + Windows + CPU, where PyTorch has no usable GPU backend, so diffusion *training* is infeasible there. This notebook runs on a **free Colab/Kaggle T4 GPU** ($0) to do the one thing your laptop can't: actually **fine-tune** the model.

**Strategy:** train a small **LoRA adapter** (a few MB) on top of frozen SD v1.5, on real pathology images from the open `flaviagiammarino/path-vqa` dataset. Then **download the adapter** and load it in the local app for CPU inference.

**Honesty:** every metric below (training loss, FID) is produced by code that actually runs. Free-GPU + small-subset + short training = a *demonstration of real fine-tuning*, not a clinical-grade model. Reported as such.

---
### How to run
1. Open this notebook in **Google Colab** (https://colab.research.google.com → Upload).
2. `Runtime → Change runtime type → T4 GPU`.
3. `Runtime → Run all`. Total time on a T4: ~30–50 min.
4. At the end, **download `multimedai_lora.safetensors`** and place it in your repo at `weights/lora/`.

## 1. Confirm GPU (this notebook REQUIRES a GPU — that's the whole point)

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2. Install pinned deps + grab the official diffusers LoRA training script

In [ ]:
!pip -q install diffusers==0.31.0 transformers==4.46.0 accelerate==1.6.0 datasets==3.0.1 peft==0.13.2 pytorch-fid==0.3.0 safetensors==0.5.3
# official LoRA text-to-image trainer matching diffusers 0.31.0
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/text_to_image/train_text_to_image_lora.py -O train_lora.py
print('ready')

## 3. Build a small image+caption dataset from PathVQA

PathVQA has no captions, only Q/A. We derive weak captions from open-ended answers: `"histopathology image, H&E stain, showing {answer}"`. We keep ~800 non-trivial examples. This is a deliberately simple caption scheme — documented, not hidden.

In [ ]:
import os, json
from datasets import load_dataset

N = 800            # subset size (free-GPU friendly)
RES = 512          # SD v1.5 native
os.makedirs('train_data', exist_ok=True)

ds = load_dataset('flaviagiammarino/path-vqa', split='train')
rows, seen = [], set()
for ex in ds:
    ans = ex['answer'].strip().lower()
    if ans in ('yes', 'no') or len(ans) < 3 or len(ans) > 60:
        continue                       # skip yes/no + degenerate answers
    # dedup images
    h = hash(ex['image'].convert('L').resize((64,64)).tobytes())
    if h in seen:
        continue
    seen.add(h)
    i = len(rows)
    fn = f'{i:05d}.jpg'
    ex['image'].convert('RGB').resize((RES, RES)).save(f'train_data/{fn}', quality=90)
    rows.append({'file_name': fn, 'text': f'histopathology image, H&E stain, showing {ans}'})
    if len(rows) >= N:
        break

with open('train_data/metadata.jsonl', 'w') as f:
    for r in rows:
        f.write(json.dumps(r) + '\n')
print(f'Built {len(rows)} image/caption pairs. Example:', rows[0])

## 4. Fine-tune the LoRA adapter (real training on GPU)

`rank=8` LoRA on the UNet attention layers. ~1000 steps at 512px fits comfortably in a free T4 session. The training loss printed here is **real**.

In [ ]:
!accelerate launch --mixed_precision=fp16 train_lora.py \
  --pretrained_model_name_or_path=runwayml/stable-diffusion-v1-5 \
  --train_data_dir=train_data \
  --caption_column=text \
  --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=1e-4 --lr_scheduler=cosine --lr_warmup_steps=0 \
  --rank=8 \
  --seed=42 \
  --output_dir=lora_out \
  --checkpointing_steps=500 \
  --validation_prompt='histopathology image, H&E stain, showing adenocarcinoma' \
  --report_to=tensorboard

## 5. Generate samples WITH vs WITHOUT the LoRA (visual proof the fine-tune did something)

In [ ]:
from diffusers import StableDiffusionPipeline
import torch, matplotlib.pyplot as plt

pipe = StableDiffusionPipeline.from_pretrained('runwayml/stable-diffusion-v1-5', torch_dtype=torch.float16, safety_checker=None).to('cuda')
prompt = 'histopathology image, H&E stain, showing adenocarcinoma'
g = torch.Generator('cuda').manual_seed(42)
base = pipe(prompt, num_inference_steps=30, generator=g).images[0]

pipe.load_lora_weights('lora_out')          # apply the fine-tuned adapter
g = torch.Generator('cuda').manual_seed(42)
tuned = pipe(prompt, num_inference_steps=30, generator=g).images[0]

fig, ax = plt.subplots(1, 2, figsize=(10,5))
ax[0].imshow(base);  ax[0].set_title('Base SD v1.5');        ax[0].axis('off')
ax[1].imshow(tuned); ax[1].set_title('+ MultiMedAI LoRA');   ax[1].axis('off')
plt.show()

## 6. Real FID: fine-tuned generations vs real pathology images

In [ ]:
import os, torch
from pytorch_fid.fid_score import calculate_fid_given_paths
os.makedirs('fid_real', exist_ok=True); os.makedirs('fid_gen', exist_ok=True)

# 50 real images already saved in train_data
import glob, shutil
for i, p in enumerate(sorted(glob.glob('train_data/*.jpg'))[:50]):
    shutil.copy(p, f'fid_real/{i:03d}.jpg')

# 50 fine-tuned generations
for i in range(50):
    g = torch.Generator('cuda').manual_seed(1000+i)
    img = pipe(prompt, num_inference_steps=25, generator=g).images[0]
    img.resize((299,299)).save(f'fid_gen/{i:03d}.jpg')

fid = calculate_fid_given_paths(['fid_real','fid_gen'], batch_size=16, device='cuda', dims=2048)
print(f'REAL FID (fine-tuned, 50 vs 50, GPU) = {fid:.3f}')
print('NOTE: small-sample FID is high-variance; compare against the CPU base-model FID from synthesis.py.')

## 7. Export the adapter — DOWNLOAD this file into your repo at `weights/lora/`

In [ ]:
import shutil, os
src = 'lora_out/pytorch_lora_weights.safetensors'
dst = 'multimedai_lora.safetensors'
shutil.copy(src, dst)
print('Size (MB):', round(os.path.getsize(dst)/1e6, 2))
try:
    from google.colab import files
    files.download(dst)        # browser download
except Exception:
    print('Not on Colab — download', dst, 'manually.')

## Done

1. Place `multimedai_lora.safetensors` in your repo: `weights/lora/multimedai_lora.safetensors`.
2. The local app's Synthesis tab will detect it and apply it during CPU inference (`pipe.load_lora_weights(...)`).
3. You now have a **genuinely fine-tuned** diffusion adapter — trained on a real GPU, applied locally on CPU. Document the FID comparison (base vs fine-tuned) in DEFENSE.md.